In [3]:
import pandas as pd
import numpy as np

DATA_PATH = r"D:\Course\python\energy forecasting\preprocessing\evaluation_test_30d.csv"

df = pd.read_csv(DATA_PATH)

df["time"] = pd.to_datetime(df["time"])
df = df.sort_values("time").reset_index(drop=True)

print("Dataset shape:", df.shape)
df.head()

Dataset shape: (1440, 27)


,time,load,solar,wind,wind_onshore,wind_offshore,hour,day_of_week,day_of_month,month,...,weather_code,temperature_lag_1,humidity_lag_1,temperature_lag_24,humidity_lag_24,load_lag_1,load_lag_24,load_lag_168,rolling_mean_24,rolling_std_24
0,2020-08-02 00:00:00+00:00,35504.0,0.0,4735.0,3459.0,1276.0,0,6,2,8,...,3,21.9,43.0,16.1,81.0,36714.0,38943.0,34507.0,44359.208333,5080.944315
1,2020-08-02 01:00:00+00:00,34628.0,0.0,4123.0,3157.0,966.0,1,6,2,8,...,3,22.3,45.0,15.0,86.0,35504.0,38305.0,34176.0,44206.000000,5321.194366
2,2020-08-02 02:00:00+00:00,34433.0,0.0,4054.0,3224.0,830.0,2,6,2,8,...,3,21.6,47.0,14.4,89.0,34628.0,37995.0,34212.0,44057.583333,5546.851304
3,2020-08-02 03:00:00+00:00,34717.0,20.0,3833.0,3467.0,366.0,3,6,2,8,...,3,21.5,48.0,14.1,91.0,34433.0,38355.0,34294.0,43906.000000,5755.260150
4,2020-08-02 04:00:00+00:00,34680.0,696.0,3769.0,3595.0,173.0,4,6,2,8,...,3,21.3,50.0,14.1,91.0,34717.0,39717.0,35397.0,43696.125000,6001.245018


In [ ]:
df.columns

Index(['time', 'load', 'solar', 'wind', 'wind_onshore', 'wind_offshore',
       'hour', 'day_of_week', 'day_of_month', 'month', 'is_weekend',
       'is_holiday', 'load_forecast', 'temperature', 'humidity', 'wind_speed',
       'precipitation', 'weather_code', 'temperature_lag_1', 'humidity_lag_1',
       'temperature_lag_24', 'humidity_lag_24', 'load_lag_1', 'load_lag_24',
       'load_lag_168', 'rolling_mean_24', 'rolling_std_24'],
      dtype='object')

: 

In [6]:
print("Start date:", df["time"].min())
print("End date:", df["time"].max())
print("Total rows:", len(df))

Start date: 2015-01-08 07:00:00+00:00
End date: 2020-09-30 23:00:00+00:00
Total rows: 50225


In [7]:
last_30_days = df[df["time"] >= df["time"].max() - pd.Timedelta(days=30)]

print("Last 30 days shape:", last_30_days.shape)

last_30_days.to_csv("test_last_30_days.csv", index=False)

Last 30 days shape: (721, 27)


In [8]:
last_90_days = df[df["time"] >= df["time"].max() - pd.Timedelta(days=90)]

print("Last 90 days shape:", last_90_days.shape)

last_90_days.to_csv("test_last_90_days.csv", index=False)

Last 90 days shape: (2161, 27)


In [9]:
window_size = 60 * 24  # 60 days hourly

start_idx = np.random.randint(0, len(df) - window_size)

random_window = df.iloc[start_idx:start_idx + window_size]

print("Random window shape:", random_window.shape)

random_window.to_csv("test_random_60_days.csv", index=False)

Random window shape: (1440, 27)


In [10]:
LOOKBACK = 720

def create_lookback_windows(df, num_samples=5):
    samples = []
    
    for i in range(num_samples):
        start_idx = np.random.randint(0, len(df) - LOOKBACK)
        sample = df.iloc[start_idx:start_idx + LOOKBACK]
        
        filename = f"test_lookback_{i+1}.csv"
        sample.to_csv(filename, index=False)
        
        samples.append(filename)
        
    return samples

files_created = create_lookback_windows(df, num_samples=5)

print("Created files:", files_created)

Created files: ['test_lookback_1.csv', 'test_lookback_2.csv', 'test_lookback_3.csv', 'test_lookback_4.csv', 'test_lookback_5.csv']


In [11]:
LOOKBACK = 720
STEP = 168  # Move by 1 week

rolling_files = []

for start in range(0, len(df) - LOOKBACK, STEP):
    sample = df.iloc[start:start + LOOKBACK]
    
    filename = f"rolling_test_{start}.csv"
    sample.to_csv(filename, index=False)
    
    rolling_files.append(filename)

print("Total rolling test files:", len(rolling_files))

Total rolling test files: 295


In [12]:
LOOKBACK = 720
MAX_LAG = 168

REQUIRED_ROWS = LOOKBACK + MAX_LAG

def create_valid_lookback(df, num_samples=5):
    samples = []

    for i in range(num_samples):
        start_idx = np.random.randint(0, len(df) - REQUIRED_ROWS)
        sample = df.iloc[start_idx:start_idx + REQUIRED_ROWS]

        filename = f"test_valid_lookback_{i+1}.csv"
        sample.to_csv(filename, index=False)

        samples.append(filename)

    return samples

files = create_valid_lookback(df, 3)
print(files)

['test_valid_lookback_1.csv', 'test_valid_lookback_2.csv', 'test_valid_lookback_3.csv']


Forecast dataset saved: (720, 27)
Evaluation dataset saved: (744, 27)
7D evaluation dataset saved: (888, 27)
30D evaluation dataset saved: (1440, 27)


In [17]:
import pandas as pd

df = pd.read_csv(DATA_PATH)

df["time"] = pd.to_datetime(df["time"])
df = df.sort_values("time").reset_index(drop=True)

LOOKBACK = 720

# Add 200 extra rows for safe lag drop
forecast_df = df.tail(LOOKBACK + 200).copy()

forecast_df.to_csv("forecast_test.csv", index=False)

print("New forecast dataset shape:", forecast_df.shape)

New forecast dataset shape: (920, 27)


In [3]:
import pandas as pd

df = pd.read_csv(DATA_PATH)

df["time"] = pd.to_datetime(df["time"])
df = df.sort_values("time").reset_index(drop=True)

LOOKBACK = 720

# Need enough rows for 30D evaluation
rows_needed = LOOKBACK + 720 + 200   # extra buffer for dropna

eval_30d_df = df.tail(rows_needed).copy()
eval_30d_df.to_csv("evaluation_test_30d_big.csv", index=False)

print("Saved 30D evaluation dataset:", eval_30d_df.shape)

Saved 30D evaluation dataset: (1640, 27)


In [ ]:

# link https://energy-forecast-api-sfrz.onrender.com/run_model